# PatchTune-mini — GRPO training (Colab)

**Lexsi AI Safety & Alignment Hackathon · Track 3 (AgentTune + AuditKIT)**
**Problem Statements #1 Reliable tool calling + #2 Efficient trajectories**
**Base model: `CohereLabs/tiny-aya-global`** (Rules §3: dense open-weights Cohere Labs LLM)

Two training paths live in `train_grpo.py`:
- **PATH A (preferred): AgentTune** `create_agentic_trainer(algorithm="grpo")` — the track toolkit.
- **PATH B (fallback): TRL `GRPOTrainer`** — used automatically if agenttune is not installable or its API fails.

Both optimise the same reward: `patchtune.reward.reward_func`.
⚠️ AgentTune GRPO rewards receive **flattened completion strings**, not dicts — `patchtune.reward.parse_rollout` handles that convention.
Deadline: **Mon Nov 16, 2026, 17:59 Dhaka**. Do not start a new run after Sun evening.

In [ ]:
# 0. GPU check
!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# 1. Install dependencies
# agenttune / auditkit are the Lexsi track toolkits (LSAL source-available licences).
!pip -q install agenttune auditkit
!pip -q install trl "transformers>=4.46,<5" datasets peft accelerate "pandas<2.3"
print("installs attempted — if agenttune failed above, PATH B (TRL) will be used automatically")

In [ ]:
# 2. Get the project (public repo, pushed during the hackathon weekend)
!git clone https://github.com/Ahmad-glowex/patchtune-lexsi.git
%cd patchtune-lexsi
import os, sys
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
# 2b. Hugging Face login — token comes from Colab Secrets (key icon, left sidebar:
# name HF_TOKEN, notebook access ON). The token is never printed.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')
print('HF token set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# 3. Generate data (150 train / 50 held-out) and sanity-check the environment end-to-end (mock eval, no model needed)
!python -m patchtune.tasks --out-dir data --train 150 --heldout 50
!python -m patchtune.evaluate --model mock-base  --tasks data/heldout.jsonl --out results/eval_mock_base.json
!python -m patchtune.evaluate --model mock-tuned --tasks data/heldout.jsonl --out results/eval_mock_tuned.json

## 4. SMOKE TEST FIRST (Fri evening de-risk)
Five steps only. Success = one path completes, a checkpoint is written, nothing crashes.
If PATH A errors (import/API), the script prints the failure and falls back to PATH B — that is expected behaviour, not a bug.

In [ ]:
# 4. Smoke test (5 steps, PATH A auto -> PATH B fallback)
!python train_grpo.py --smoke --out runs/smoke

## 5. Full GRPO run (Sun)
Keep generations/steps small — free Colab T4 is the binding constraint (research-report R2).
Save checkpoints to Drive/HF as you go; Colab sessions drop.

In [ ]:
# 5. Full training run
!python train_grpo.py --steps 300 --out runs/patchtune-grpo

## 6. Evaluate base vs tuned (identical held-out set)
Report at least: task success, tool-call accuracy, calls/task, malformed rate.
- The **real-model** rollout loop runs here interactively (generation interleaved with `ToolExecutor.dispatch` in a sandbox per task), then traces are scored with `patchtune.reward` metrics **and** passed through **AuditKIT** agent scorers (tool-call F1 / validity / redundant calls / loop detection) so both track toolkits are chained in the pipeline.
- If AuditKIT integration blocks, keep its error, run the built-in metrics, and note it in the article's experiments section — do not silently drop it.

In [ ]:
# 6. Built-in metrics on the mock harness stay as the pipeline sanity reference.
# Real-model eval: run the interactive rollout (see README §Evaluation) for
#   base  -> results/eval_base.json
#   tuned -> results/eval_tuned.json   (adapter = runs/patchtune-grpo)
# Then build the before/after table:
import json
for name in ("base", "tuned"):
    try:
        s = json.load(open(f"results/eval_{name}.json"))
        print(name, {k: s[k] for k in ("task_success_rate", "tool_call_accuracy", "avg_calls_per_task", "malformed_rate")})
    except FileNotFoundError:
        print(name, "-> not run yet")

## 7. Publish artifacts (all public, un-gated)
- **HF model collection**: base reference + trained adapter (`runs/patchtune-grpo`).
- **HF dataset collection**: `data/train.jsonl`, `data/heldout.jsonl`, rollout traces, eval JSONs.
- Fill in your HF username below. Check Tiny Aya's licence on its model card and quote it accurately in the article.

In [ ]:
# 7. Upload to Hugging Face (fill in HF_USER first)
HF_USER = "TODO-your-hf-username"
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(folder_path="runs/patchtune-grpo", repo_id=f"{HF_USER}/patchtune-mini-aya-grpo", repo_type="model")
# api.upload_folder(folder_path="data", repo_id=f"{HF_USER}/patchtune-mini-data", repo_type="dataset")
print("Set HF_USER and uncomment to upload.")

## 8. Submit (Mon, target 15:00 Dhaka, hard stop 17:59)
At https://hackathon.lexsi.ai/submission/ with: registered email, abstract (<500 words), HF model collection link, HF dataset collection link, HF Article (9 checklist sections) or GitHub README, GitHub repo, track = Agentic Fine-tuning, problem statements #1 + #2. Latest submission before the deadline counts.